In [5]:
import os
import csv
import psycopg2

from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [41]:
import pandas as pd
import numpy as np
import uuid
import random
from datetime import datetime, timedelta

np.random.seed(42)


# ------------------------
# CONFIG
# ------------------------
NUM_USERS = 2000
NUM_ISSUERS = 200
NUM_TOKENS = 155
NUM_TRANSACTIONS = 50000

START_DATE = datetime(2025, 1, 1)

PRICE_INCREMENT = 0.000023
MAX_PRICE = 24.0

def generate_uuid():
    return str(uuid.uuid4())

def random_timestamp():
    return START_DATE + timedelta(days=random.randint(0, 365))

In [ ]:
# from io import StringIO

# from psycopg2 import sql

# # Get the connection string from the environment variable
# conn_string = os.getenv("DATABASE_URL")


# # def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
# #     """Bulk-load a DataFrame into Postgres via COPY.

# #     By default clears the table first with ``TRUNCATE ... RESTART IDENTITY`` so each
# #     run replaces rows (avoids duplicate key errors on reload).

# #     If other tables reference this one, plain ``TRUNCATE`` may fail; set
# #     ``truncate_cascade=True`` to truncate dependent tables too (destructive).

# #     Set ``clear_first=False`` to append without clearing.
# #     """
# #     buf = StringIO()
# #     df.to_csv(buf, index=False, header=True)
# #     buf.seek(0)
# #     with psycopg2.connect(conn_string) as conn:
# #         with conn.cursor() as cur:
# #             cur.execute(sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY CASCADE" ).format(sql.Identifier(table_name)))
# #             #         )
# #             # if clear_first:
# #             #     if truncate_cascade:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY CASCADE"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             #     else:
# #             #         cur.execute(
# #             #             sql.SQL(
# #             #                 "TRUNCATE TABLE {} RESTART IDENTITY"
# #             #             ).format(sql.Identifier(table_name))
# #             #         )
# #             copy_sql = sql.SQL(
# #                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
# #             ).format(sql.Identifier(table_name))
# #             cur.copy_expert(copy_sql, buf)
# #         conn.commit()
# #     print(f"Loaded {len(df)} rows into {table_name!r}")

# def load_data(conn_string, df, table_name, *, clear_first=True, truncate_cascade=False):
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)

#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             # 1. Properly execute the TRUNCATE
#             if clear_first:
#                 # Add CASCADE if requested
#                 suffix = " CASCADE" if truncate_cascade else ""
#                 truncate_stmt = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
#                     sql.Identifier(table_name)
#                 )
#                 cur.execute(truncate_stmt) # This sends the command to the DB

#             # 2. Perform the COPY
#             copy_sql = sql.SQL(
#                 "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#             ).format(sql.Identifier(table_name))
#             cur.copy_expert(copy_sql, buf)
            
#         # 3. Commit the transaction (essential!)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")


In [12]:
import os
import psycopg2
from psycopg2 import sql
from io import StringIO
conn_string = os.getenv("DATABASE_URL")

def truncate_table(conn_string, table_name, cascade=False):
    """Forcefully clears the table and resets identities."""
    suffix = " CASCADE" if cascade else ""
    query = sql.SQL("TRUNCATE TABLE {} RESTART IDENTITY" + suffix).format(
        sql.Identifier(table_name)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
        conn.commit()
    print(f"Table {table_name!r} truncated successfully.")

# def copy_data(conn_string, df, table_name):
#     """Loads DataFrame into the table via COPY."""
#     buf = StringIO()
#     df.to_csv(buf, index=False, header=True)
#     buf.seek(0)
    
#     copy_sql = sql.SQL(
#         "COPY {} FROM STDIN WITH (FORMAT csv, HEADER true)"
#     ).format(sql.Identifier(table_name))
    
#     with psycopg2.connect(conn_string) as conn:
#         with conn.cursor() as cur:
#             cur.copy_expert(copy_sql, buf)
#         conn.commit()
#     print(f"Loaded {len(df)} rows into {table_name!r}")

def copy_data(conn_string, df, table_name):
    buf = StringIO()
    df.to_csv(buf, index=False, header=True)
    buf.seek(0)
    
    # Identify the columns from the DataFrame
    columns = [sql.Identifier(col) for col in df.columns]
    
    # Format: COPY table_name (col1, col2, ...) FROM STDIN ...
    copy_sql = sql.SQL(
        "COPY {} ({}) FROM STDIN WITH (FORMAT csv, HEADER true)"
    ).format(
        sql.Identifier(table_name),
        sql.SQL(', ').join(columns)
    )
    
    with psycopg2.connect(conn_string) as conn:
        with conn.cursor() as cur:
            cur.copy_expert(copy_sql, buf)
        conn.commit()
    print(f"Loaded {len(df)} rows into {table_name!r}")


In [ ]:
# ------------------------
# USERS
# ------------------------
users = []

for _ in range(NUM_USERS):
    user_id = generate_uuid()
    users.append({
        "user_id": user_id,
        "user_role": random.choice(["FAN", "ISSUER", "BOTH"]),
        "display_name": f"user_{random.randint(1000,9999)}",
        "username": f"user_{1000 + _}",
        # "username": f"user_{random.randint(10000,99999)}",
        "referral_id": None,
        "email": f"user{10000 + _}@test.com",
        # "email": f"user{random.randint(10000,99999)}@test.com",
        "email_verified": random.choice([True, False]),
        "email_verification_status": random.choice(["PENDING","VERIFIED"]),
        "email_verification_score": round(random.uniform(50,100),2),
        "country": random.choice(["US","CA","UK"]),
        "zip_code": str(random.randint(10000,99999)),
        "gender": random.choice(["Male","Female","Other"]),
        "time_zone": "UTC",
        # "login_methods_allowed": ["EMAIL_PASSWORD"],
        "password_hash": "hash",
        "passkey_public_key": None,
        "primary_social_platform": None,
        "primary_social_handle": None,
        "mfa_enabled": False,
        "account_status": "ACTIVE",
        "created_at": random_timestamp(),
        "updated_at": None,
        "last_login_at": None,
        "created_ip": "127.0.0.1",
        "created_user_agent": "generator",
        "metadata": None
    })
users_df = pd.DataFrame(users)
users_df.to_csv("./users.csv", index=False)
truncate_table(conn_string, 'users', cascade=True)
copy_data(conn_string, users_df,'users')


Table 'users' truncated successfully.
Loaded 2000 rows into 'users'


In [ ]:
# ------------------------
# ISSUERS + VERIFICATION LOGIC
# ------------------------
issuers = []
identity = []
social = []

platforms = ["YOUTUBE", "INSTAGRAM", "X"]

issuer_users = users_df.sample(NUM_ISSUERS).reset_index(drop=True)

# First 150 → fully PASSED
PASSED_COUNT = 150

for i, row in issuer_users.iterrows():
    issuer_id = generate_uuid()

    # ---------------- STATUS CONTROL ----------------
    if i < PASSED_COUNT:
        id_status = "PASSED"
        social_status = "PASSED"
        issuer_status = "PASSED"
    else:
        # force at least one failure/pending
        id_status = random.choice(["FAILED", "PENDING"])
        social_status = random.choice(["FAILED", "PENDING"])

        issuer_status = "PENDING" if "PENDING" in [id_status, social_status] else "FAILED"

    # ---------------- ISSUER ----------------
    issuers.append({
        "issuer_id": issuer_id,
        "user_id": row["user_id"],
        "issuer_type": random.choice(["ATHLETE"]),#, "CREATOR"]),
        "email": row["email"],
        "username": row["username"],
        "status": issuer_status,
        "level": random.choice(["YOUTH","COLLEGE","PRO"]),
        "country": row["country"],
        "region": "NA",
        "created_at": row["created_at"],
        "updated_at": None,
        "metadata": None
    })

    # ---------------- IDENTITY ----------------
    identity.append({
        "identity_check_id": generate_uuid(),
        "issuer_id": issuer_id,
        "provider": "Persona",
        "status": id_status,
        "level": "basic",
        "alias_confidence": round(random.uniform(70,100),2),
        "opted_in": True,
        "initiated_at": random_timestamp(),
        "completed_at": random_timestamp(),
        "failure_reason": None if id_status == "PASSED" else "check_failed",
        "raw_response": None
    })

    # ---------------- SOCIAL ----------------
    for p in random.sample(platforms, k=1):
        social.append({
            "social_verif_id": generate_uuid(),
            "issuer_id": issuer_id,
            "platform": p,
            "handle": f"{p.lower()}_{random.randint(1000,9999)}",
            "followers_count": random.randint(1000,1000000),
            "verified": True if social_status == "PASSED" else False,
            "initiated_at": random_timestamp(),
            "completed_at": random_timestamp(),
            "attempts": random.randint(1,3),
            "status": social_status,
            "metadata": None
        })

# Convert to DataFrames
issuers_df = pd.DataFrame(issuers)
identity_df = pd.DataFrame(identity)
social_df = pd.DataFrame(social)

issuers_df.to_csv("./issuers.csv", index=False)
identity_df.to_csv("./identity_verification.csv", index=False)
social_df.to_csv("./social_verification.csv", index=False)

truncate_table(conn_string, 'issuers', cascade=True)
copy_data(conn_string, issuers_df,'issuers')

truncate_table(conn_string, 'identity_verification', cascade=True)
copy_data(conn_string, identity_df,'identity_verification')

truncate_table(conn_string, 'social_verification', cascade=True)
copy_data(conn_string, social_df,'social_verification')

Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'
Table 'identity_verification' truncated successfully.
Loaded 200 rows into 'identity_verification'
Table 'social_verification' truncated successfully.
Loaded 200 rows into 'social_verification'


In [ ]:
# # ------------------------
# # ISSUERS
# # ------------------------
# issuer_users = users_df.sample(NUM_ISSUERS)
# issuers = []

# for _, row in issuer_users.iterrows():
#     issuer_id = generate_uuid()
#     issuers.append({
#         "issuer_id": issuer_id,
#         "user_id": row["user_id"],
#         "issuer_type": random.choices(["ATHLETE","CREATOR"],weights=[8,1])[0],
#         "email": row["email"],
#         "username": row["username"],
#         "status" : random.choices(["PENDING","PASSED","FAILED"],weights=[2,7.5,0.5])[0],
#         "level": random.choice(["YOUTH","COLLEGE","PRO"]),
#         "country": row["country"],
#         "region": "NA",
#         "created_at": row["created_at"],
#         "updated_at": None,
#         "metadata": None
#     })
# issuers_df = pd.DataFrame(issuers)
# issuers_df.to_csv("./issuers.csv", index=False)
# truncate_table(conn_string, 'issuers', cascade=True)
# copy_data(conn_string, issuers_df,'issuers')



Table 'issuers' truncated successfully.
Loaded 200 rows into 'issuers'


In [53]:
# ------------------------
# ATHLETE PROFILE
# ------------------------
athletes = []
for _, row in issuers_df.iterrows():
    if row["issuer_type"] == "ATHLETE":
        athletes.append({
            "issuer_id": row["issuer_id"],
            "user_id": row["user_id"],
            "sport": random.choice(["Basketball","Soccer","Football"]),
            "team": f"Team_{random.randint(1,20)}",
            "league": "Youth",
            "position_primary": "Forward",
            "position_secondary": None,
            "multi_sport": False,
            "biography": "bio",
            "profile_completion": random.randint(50,100),
            "created_at": random_timestamp(),
            "updated_at": None,
            "metadata": None
        })
athlete_df = pd.DataFrame(athletes)

athlete_df.to_csv("./athletes.csv", index=False)

truncate_table(conn_string, 'athlete_profile', cascade=True)
copy_data(conn_string, athlete_df,'athlete_profile')

Table 'athlete_profile' truncated successfully.
Loaded 200 rows into 'athlete_profile'


In [ ]:
# ------------------------
# POST SIGNUP 
# ------------------------
post_signup = []

for _, row in issuers_df.iterrows():
    post_signup.append({
        "issuer_id": row["issuer_id"],
        "wallet_provisioned": random.choice([True, False]),
        "wallet_address": f"0x{uuid.uuid4().hex[:40]}",
        "wallet_email_sent": True,
        "dashboard_redirect": True,
        "oauth_verified_min2": random.choice([True, False]),
        "verified_at": random_timestamp(),
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

post_signup_df = pd.DataFrame(post_signup)

post_signup_df.to_csv("./athletes.csv", index=False)

truncate_table(conn_string, 'issuer_post_signup', cascade=True)
copy_data(conn_string, post_signup_df,'issuer_post_signup')

Table 'issuer_post_signup' truncated successfully.
Loaded 200 rows into 'issuer_post_signup'


In [ ]:
# ------------------------
# ISSUER PREFERENCES 
# ------------------------
preferences = []

for _, row in issuers_df.iterrows():
    preferences.append({
        "issuer_id": row["issuer_id"],
        "raise_target_usd": random.randint(10000,1000000),
        "token_supply_goal": random.randint(100000,1000000),
        "enable_referrals": False,
        "notification_prefs": None,
        "created_at": random_timestamp(),
        "updated_at": None,
        "metadata": None
    })

preferences_df = pd.DataFrame(preferences)

preferences_df.to_csv("./issuer_preferences.csv", index=False)

truncate_table(conn_string, 'issuer_preferences', cascade=True)
copy_data(conn_string, preferences_df,'issuer_preferences')

Table 'issuer_preferences' truncated successfully.
Loaded 200 rows into 'issuer_preferences'
